# 00. EDA 템플릿

당일 데이터만 갈아끼우면 바로 돌아가도록 미리 작성한 EDA 노트북입니다.
정형/음향 공통 점검 항목을 담았어요. **STEP 1의 경로만 수정**하세요.

## STEP 0. import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
print('ready')

## STEP 1. 데이터 로드 (여기만 수정)

In [ ]:
# train = pd.read_csv('../data/train.csv')
# test  = pd.read_csv('../data/test.csv')

# 데모: 합성 데이터
from sklearn.datasets import make_classification
X, y = make_classification(n_samples=1000, n_features=10, n_informative=6, n_classes=3,
                           n_clusters_per_class=1, random_state=SEED)
train = pd.DataFrame(X, columns=[f'f{i}' for i in range(X.shape[1])])
train['target'] = y
print('shape:', train.shape)
train.head()

## STEP 2. 기본 점검 — 결측치·타입·기초통계

In [ ]:
miss = train.isnull().sum()
miss = miss[miss > 0]
print('=== 결측치 ===')
print(miss if len(miss) else '결측치 없음')
print('\n=== 데이터 타입 ===')
print(train.dtypes.value_counts())
print('\n=== 기초 통계 ===')
train.describe().T.head(10)

## STEP 3. 타깃 분포 (불균형 확인)

In [ ]:
tc = train['target'].value_counts()
print(tc)
imbalance = tc.max() / tc.min()
print(f'\n불균형 비율: {imbalance:.2f}배')
if imbalance > 3:
    print('⚠️ 불균형 → class_weight, stratified split 고려')
tc.plot(kind='bar', title='Target distribution'); plt.show()

## STEP 4. 특징 분포 & 상관

In [ ]:
feat_cols = [c for c in train.columns if c != 'target']
train[feat_cols].hist(bins=30, figsize=(12, 8)); plt.tight_layout(); plt.show()

In [ ]:
corr = train[feat_cols].corr()
plt.figure(figsize=(8,6)); plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(); plt.title('Feature correlation'); plt.show()
# 상관 높은 쌍 (다중공선성 점검)
high = [(feat_cols[i], feat_cols[j], corr.iloc[i,j])
        for i in range(len(feat_cols)) for j in range(i+1, len(feat_cols))
        if abs(corr.iloc[i,j]) > 0.8]
print('상관 0.8 초과 쌍:', high or '없음')

## STEP 5. 누수 점검 체크리스트

- [ ] train/test 분포가 비슷한가? (covariate shift)
- [ ] 그룹 단위(녹음 ID, 환자 ID) 누수 가능성? → GroupKFold
- [ ] target으로 만든 feature가 섞이지 않았나?
- [ ] 시계열이면 시간 순서를 지켰나?

> 음향 데이터라면 `audio_eda` 노트북의 샘플레이트·길이·스펙트로그램 점검을 추가로 수행하세요.